# Cloud Platform Cost — Standalone Mock Data Loader

Loads 90 days of Azure / AWS / GCP mock billing data into
`<catalog>.platform.cloud_platform_costs`.

**Run all three cells in order.** Estimated time: ~3 minutes.

Configure via **Databricks widgets** (set when running the notebook) or **environment variables**:
- `DATABRICKS_WAREHOUSE_ID` — SQL warehouse to use
- `UC_CATALOG` — Unity Catalog catalog name (e.g. `workspace`)
- `RECREATE_TABLE` — set to `true` to drop and recreate the table (default: `false`)

| Cell | What it does |
|------|-------------|
| 1 | Read config from widgets/env vars, define helpers |
| 2 | Generate ~55 K mock rows (Azure 64 SKUs, AWS 85 SKUs, GCP 80 SKUs) |
| 3 | Create Delta table + INSERT in 300-row chunks |


In [ ]:
# ── CONFIG — provide via Databricks widgets or env vars ────────
# Required: DATABRICKS_WAREHOUSE_ID, UC_CATALOG
# Optional: RECREATE_TABLE (set to 'true' to drop and recreate; default false)
# ────────────────────────────────────────────────────────────────────────────

import os, time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

def _get_config(widget_name, env_name):
    try:
        dbutils.widgets.text(widget_name, '')
        value = dbutils.widgets.get(widget_name).strip()
        if value:
            return value
    except:
        pass
    return os.environ.get(env_name, '').strip()

WAREHOUSE_ID   = _get_config('DATABRICKS_WAREHOUSE_ID', 'DATABRICKS_WAREHOUSE_ID')
CATALOG        = _get_config('UC_CATALOG', 'UC_CATALOG')
RECREATE_TABLE = _get_config('RECREATE_TABLE', 'RECREATE_TABLE').lower() == 'true'

if not WAREHOUSE_ID:
    raise ValueError('Missing required configuration: DATABRICKS_WAREHOUSE_ID')
if not CATALOG:
    raise ValueError('Missing required configuration: UC_CATALOG')

try:
    ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    HOST  = ctx.apiUrl().get()
    TOKEN = ctx.apiToken().get()
except:
    HOST  = os.environ.get('DATABRICKS_HOST', '')
    TOKEN = os.environ.get('DATABRICKS_TOKEN', '')

client = WorkspaceClient(host=HOST, token=TOKEN)

def run_sql(label, sql, timeout_sec=300):
    resp = client.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID, statement=sql, wait_timeout='50s')
    deadline = time.time() + timeout_sec
    while resp.status.state in (StatementState.PENDING, StatementState.RUNNING):
        if time.time() > deadline:
            print(f'  TIMEOUT: {label}'); return False
        time.sleep(3)
        resp = client.statement_execution.get_statement(resp.statement_id)
    if resp.status.state == StatementState.SUCCEEDED:
        print(f'  OK: {label}'); return True
    err = getattr(resp.status.error, 'message', str(resp.status.state))
    print(f'  WARN ({err[:100]}): {label}'); return False

def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

print(f'Ready  warehouse={WAREHOUSE_ID}  catalog={CATALOG}  recreate_table={RECREATE_TABLE}')


In [ ]:
import random, json as _json
from datetime import datetime, timedelta

AZURE_SUBSCRIPTIONS = [
    ('sub-prod-001',  'Production',    'eastus'),
    ('sub-dev-002',   'Development',   'westeurope'),
    ('sub-data-003',  'Data Platform', 'eastus2'),
]
AWS_ACCOUNTS = [
    ('111122223333', 'corp-prod-aws',  'us-east-1'),
    ('444455556666', 'corp-dev-aws',   'us-west-2'),
    ('777788889999', 'corp-data-aws',  'eu-west-1'),
]
GCP_PROJECTS = [
    ('corp-prod-gcp', 'Corp Production GCP',  'us-central1'),
    ('corp-ml-gcp',   'Corp ML Platform GCP', 'us-east1'),
]

# Tags pool — realistic enterprise resource tagging
TEAMS        = ['platform-core','etl-pipeline','ml-ops','analytics-eng','feature-store',
                'data-quality','streaming-ingest','reporting','bi-team','quant-research',
                'fraud-ml','aml-detection','credit-scoring','data-governance']
ENVS         = ['prod','prod','prod','staging','dev']      # prod-weighted
COST_CENTERS = [f'CC-{i:03d}' for i in range(1, 16)]
PROJECTS     = ['project-alpha','project-beta','lakehouse-migration','ml-platform',
                'data-mesh','compliance-reporting','real-time-fraud','customer-360']

def make_tags():
    return _json.dumps({
        'team':        random.choice(TEAMS),
        'environment': random.choice(ENVS),
        'cost_center': random.choice(COST_CENTERS),
        'project':     random.choice(PROJECTS),
    })

# Regional price multipliers (relative to base)
REGION_MUL = {
    'eastus': 1.0, 'eastus2': 1.0, 'westeurope': 1.12,
    'us-east-1': 1.0, 'us-west-2': 1.02, 'eu-west-1': 1.10,
    'us-central1': 1.0, 'us-east1': 1.0,
}

# Pricing model weights for compute vs non-compute
PM_COMPUTE  = [('ON_DEMAND',0.45),('RESERVED_1Y',0.30),('RESERVED_3Y',0.15),('SPOT',0.10)]
PM_DATA     = [('ON_DEMAND',0.70),('RESERVED_1Y',0.25),('RESERVED_3Y',0.05),('SPOT',0.00)]
PM_OTHER    = [('ON_DEMAND',1.00),('RESERVED_1Y',0.00),('RESERVED_3Y',0.00),('SPOT',0.00)]
RESERVED_DISC = {'ON_DEMAND':1.0,'RESERVED_1Y':0.62,'RESERVED_3Y':0.42,'SPOT':0.20}

def pick_pm(models):
    choices, weights = zip(*[(m,w) for m,w in models if w>0])
    return random.choices(choices, weights=weights)[0]

# ─────────────────────────────────────────────────────────────────────────────
# AZURE SKU CATALOG
# Each entry: (service, category, sku_description, usage_unit, unit_price,
#              qty_min_day, qty_max_day, pm_profile)
# ─────────────────────────────────────────────────────────────────────────────
AZURE_SKUS = [
  # Virtual Machines
  ('Virtual Machines','Compute','D2s v3 Spot/On-Demand','1 Hour',0.096,24,96,PM_COMPUTE),
  ('Virtual Machines','Compute','D4s v3 Spot/On-Demand','1 Hour',0.192,24,96,PM_COMPUTE),
  ('Virtual Machines','Compute','D8s v3 Spot/On-Demand','1 Hour',0.384,12,72,PM_COMPUTE),
  ('Virtual Machines','Compute','E4s v3 (Memory Opt)','1 Hour',0.252,12,48,PM_COMPUTE),
  ('Virtual Machines','Compute','NC6s v3 (GPU)','1 Hour',3.060,4,24,PM_COMPUTE),
  # AKS
  ('Azure Kubernetes Service','Compute','Standard_D4s_v3 Node','1 Hour',0.192,72,240,PM_COMPUTE),
  ('Azure Kubernetes Service','Compute','Standard_D8s_v3 Node','1 Hour',0.384,48,168,PM_COMPUTE),
  # App Service
  ('Azure App Service','Compute','P2v3 Plan','1 Hour',0.194,24,48,PM_DATA),
  ('Azure App Service','Compute','P3v3 Plan','1 Hour',0.388,12,24,PM_DATA),
  # Functions
  ('Azure Functions','Serverless','Execution Units','1M Executions',0.20,1,50,PM_OTHER),
  ('Azure Functions','Serverless','Execution Time','GB-s',0.000016,10000,500000,PM_OTHER),
  # Container Instances
  ('Azure Container Instances','Compute','vCPU Duration','vCPU-s',0.0000135,3600,86400,PM_OTHER),
  ('Azure Container Instances','Compute','Memory Duration','GB-s',0.0000015,7200,172800,PM_OTHER),
  # Batch
  ('Azure Batch','Compute','Standard_D4s_v3','1 Hour',0.192,24,120,PM_COMPUTE),
  # Databricks
  ('Azure Databricks','Analytics','Jobs Compute DBUs','DBU',0.150,100,2000,PM_DATA),
  ('Azure Databricks','Analytics','All-Purpose Compute DBUs','DBU',0.550,50,500,PM_DATA),
  ('Azure Databricks','Analytics','SQL Compute DBUs','DBU',0.550,100,1000,PM_DATA),
  ('Azure Databricks','Analytics','DLT Core DBUs','DBU',0.200,50,400,PM_DATA),
  # Synapse
  ('Azure Synapse Analytics','Analytics','Dedicated SQL Pool DWUs','DWH-Hour',1.200,24,48,PM_DATA),
  ('Azure Synapse Analytics','Analytics','Serverless SQL Queries','TB Processed',5.00,0.1,10,PM_OTHER),
  # HDInsight
  ('Azure HDInsight','Analytics','D4 v2 Worker Node','1 Hour',0.293,48,144,PM_COMPUTE),
  # Stream Analytics
  ('Azure Stream Analytics','Analytics','Streaming Units','SU/Hour',0.080,6,24,PM_OTHER),
  # Data Explorer
  ('Azure Data Explorer','Analytics','D11 v2 Compute','1 Hour',0.359,8,24,PM_COMPUTE),
  # Blob Storage
  ('Azure Blob Storage','Storage','LRS Data Stored','GB/Month',0.018,500,20000,PM_OTHER),
  ('Azure Blob Storage','Storage','GRS Data Stored','GB/Month',0.035,100,5000,PM_OTHER),
  ('Azure Blob Storage','Storage','Read Operations','10K',0.004,100,5000,PM_OTHER),
  ('Azure Blob Storage','Storage','Write Operations','10K',0.050,50,2000,PM_OTHER),
  # Data Lake Storage
  ('Azure Data Lake Storage','Storage','LRS Hierarchical Namespace','GB/Month',0.023,1000,50000,PM_OTHER),
  ('Azure Data Lake Storage','Storage','Read Transactions','10K',0.004,500,10000,PM_OTHER),
  # Files
  ('Azure Files','Storage','LRS File Storage','GB/Month',0.060,100,2000,PM_OTHER),
  # Backup
  ('Azure Backup','Storage','LRS Backup Storage','GB/Month',0.024,500,10000,PM_OTHER),
  # SQL Database
  ('Azure SQL Database','Database','General Purpose 4 vCores','1 Hour',0.725,24,24,PM_DATA),
  ('Azure SQL Database','Database','Business Critical 8 vCores','1 Hour',2.899,24,24,PM_DATA),
  ('Azure SQL Database','Database','Serverless 1-4 vCores','vCore-Hour',0.181,2,96,PM_OTHER),
  # Cosmos DB
  ('Azure Cosmos DB','Database','400 RU/s Provisioned','100 RU/s/Hour',0.008,24,24,PM_DATA),
  ('Azure Cosmos DB','Database','Transactional Storage','GB/Month',0.250,100,5000,PM_OTHER),
  # PostgreSQL
  ('Azure Database for PostgreSQL','Database','General Purpose 4 vCores','1 Hour',0.302,24,24,PM_DATA),
  # Redis
  ('Azure Cache for Redis','Database','C2 Standard','1 Hour',0.093,24,24,PM_DATA),
  ('Azure Cache for Redis','Database','P1 Premium','1 Hour',0.554,24,24,PM_DATA),
  # Data Factory
  ('Azure Data Factory','Integration','Data Flow vCore-Hour','vCore-Hour',0.274,10,200,PM_OTHER),
  ('Azure Data Factory','Integration','Orchestration Activity Runs','1K Runs',1.00,1,50,PM_OTHER),
  ('Azure Data Factory','Integration','Data Movement','DIU-Hour',0.250,5,100,PM_OTHER),
  # Event Hubs
  ('Azure Event Hubs','Integration','Standard Throughput Units','TU-Hour',0.030,8,32,PM_OTHER),
  ('Azure Event Hubs','Integration','Ingress Events','1M Events',0.028,10,500,PM_OTHER),
  # Service Bus
  ('Azure Service Bus','Integration','Standard Operations','1M Operations',0.10,1,50,PM_OTHER),
  # Logic Apps
  ('Azure Logic Apps','Integration','Action Executions','10K Actions',0.25,1,20,PM_OTHER),
  # API Management
  ('Azure API Management','Integration','Developer Tier Calls','1M Calls',3.50,0.1,5,PM_OTHER),
  ('Azure API Management','Integration','Standard Tier Calls','1M Calls',3.50,1,50,PM_OTHER),
  # OpenAI
  ('Azure OpenAI Service','AI / ML','GPT-4 Input Tokens','1K Tokens',0.03,100,10000,PM_OTHER),
  ('Azure OpenAI Service','AI / ML','GPT-4 Output Tokens','1K Tokens',0.06,50,5000,PM_OTHER),
  ('Azure OpenAI Service','AI / ML','Embeddings Tokens','1K Tokens',0.0001,1000,100000,PM_OTHER),
  # Machine Learning
  ('Azure Machine Learning','AI / ML','Compute Cluster NC6s v3','1 Hour',3.060,4,24,PM_COMPUTE),
  ('Azure Machine Learning','AI / ML','Managed Online Endpoint','1 Hour',0.096,24,48,PM_OTHER),
  # Cognitive Services
  ('Azure Cognitive Services','AI / ML','Language Transactions','1K Transactions',1.50,1,20,PM_OTHER),
  ('Azure Cognitive Services','AI / ML','Vision Transactions','1K Transactions',1.00,1,30,PM_OTHER),
  # Networking
  ('Azure Virtual Network','Networking','Peering Data Transfer','GB',0.010,100,10000,PM_OTHER),
  ('Azure Application Gateway','Networking','WAF v2 Gateway Hours','1 Hour',0.360,24,24,PM_OTHER),
  ('Azure Application Gateway','Networking','WAF v2 Capacity Units','CU-Hour',0.008,50,200,PM_OTHER),
  ('Azure Load Balancer','Networking','Standard LB Hours','1 Hour',0.005,24,24,PM_OTHER),
  ('Azure CDN','Networking','Zone 1 Data Transfer','GB',0.087,10,1000,PM_OTHER),
  ('Azure DNS','Networking','Hosted Zone','Month',0.50,1,20,PM_OTHER),
  ('Azure ExpressRoute','Networking','Standard Circuit Port','Month',55.0,1,4,PM_OTHER),
  ('Azure ExpressRoute','Networking','Metered Data','GB',0.025,100,10000,PM_OTHER),
  # Security
  ('Azure Key Vault','Security','Secrets Operations','10K Operations',0.03,1,100,PM_OTHER),
  ('Microsoft Defender for Cloud','Security','Server Plan 2','Node/Hour',0.017,50,200,PM_OTHER),
  ('Azure Firewall','Security','Deployment Hours','1 Hour',1.25,24,24,PM_OTHER),
  ('Azure Firewall','Security','Data Processed','GB',0.016,100,5000,PM_OTHER),
  # Management
  ('Azure Monitor','Management','Log Analytics Data Ingestion','GB',2.76,10,500,PM_OTHER),
  ('Azure Monitor','Management','Metrics API Calls','1M Calls',0.01,1,50,PM_OTHER),
  ('Azure Log Analytics','Management','Data Retention','GB/Month',0.10,100,5000,PM_OTHER),
  # DevOps
  ('Azure DevOps','DevOps','Basic + Test Plans User','User/Month',52.0,10,100,PM_OTHER),
  ('Azure DevOps','DevOps','Parallel Jobs','Job/Month',40.0,2,10,PM_OTHER),
  ('Azure Container Registry','DevOps','Standard Registry','Month',20.0,1,3,PM_OTHER),
]

# ─────────────────────────────────────────────────────────────────────────────
# AWS SKU CATALOG
# ─────────────────────────────────────────────────────────────────────────────
AWS_SKUS = [
  # EC2
  ('Amazon EC2','Compute','BoxUsage:m5.xlarge','Hrs',0.192,24,96,PM_COMPUTE),
  ('Amazon EC2','Compute','BoxUsage:m5.2xlarge','Hrs',0.384,12,72,PM_COMPUTE),
  ('Amazon EC2','Compute','BoxUsage:r5.2xlarge','Hrs',0.504,12,48,PM_COMPUTE),
  ('Amazon EC2','Compute','BoxUsage:c5.4xlarge','Hrs',0.680,8,48,PM_COMPUTE),
  ('Amazon EC2','Compute','SpotUsage:m5.xlarge','Hrs',0.058,24,96,PM_COMPUTE),
  ('Amazon EC2','Compute','BoxUsage:p3.2xlarge (GPU)','Hrs',3.060,4,24,PM_COMPUTE),
  # EKS
  ('Amazon EKS','Compute','AmazonEKS-Hours','Hrs',0.100,24,24,PM_OTHER),
  ('Amazon EKS','Compute','BoxUsage:m5.xlarge (Node)','Hrs',0.192,48,192,PM_COMPUTE),
  # ECS/Fargate
  ('Amazon ECS','Compute','BoxUsage:m5.large','Hrs',0.096,24,96,PM_COMPUTE),
  ('AWS Fargate','Serverless','Fargate-vCPU-Hours','vCPU-Hours',0.04048,10,200,PM_OTHER),
  ('AWS Fargate','Serverless','Fargate-GB-Hours','GB-Hours',0.004445,20,400,PM_OTHER),
  # Lambda
  ('AWS Lambda','Serverless','Lambda-GB-Second','GB-s',0.0000166667,10000,5000000,PM_OTHER),
  ('AWS Lambda','Serverless','Lambda-Request','1M Requests',0.20,1,100,PM_OTHER),
  # Batch
  ('AWS Batch','Compute','BoxUsage:c5.2xlarge','Hrs',0.340,8,48,PM_COMPUTE),
  # S3
  ('Amazon S3','Storage','TimedStorage-ByteHrs','GB-Mo',0.023,1000,50000,PM_OTHER),
  ('Amazon S3','Storage','Requests-Tier1 (PUT/COPY)','1K Requests',0.005,10,1000,PM_OTHER),
  ('Amazon S3','Storage','Requests-Tier2 (GET)','1K Requests',0.0004,100,10000,PM_OTHER),
  ('Amazon S3','Storage','DataTransfer-Out-Bytes','GB',0.090,10,1000,PM_OTHER),
  # EBS
  ('Amazon EBS','Storage','EBS:VolumeUsage.gp3','GB-Mo',0.080,500,20000,PM_OTHER),
  ('Amazon EBS','Storage','EBS:VolumeUsage.io2','GB-Mo',0.125,100,5000,PM_OTHER),
  # EFS
  ('Amazon EFS','Storage','EFS TimedStorage-ByteHrs','GB-Mo',0.30,100,5000,PM_OTHER),
  # Glacier
  ('Amazon S3 Glacier','Storage','Glacier TimedStorage','GB-Mo',0.004,500,20000,PM_OTHER),
  # FSx
  ('Amazon FSx','Storage','FSx:StorageUsage','GB-Mo',0.13,100,2000,PM_OTHER),
  # RDS
  ('Amazon RDS','Database','RDS:db.r5.large Multi-AZ','Hrs',0.48,24,24,PM_DATA),
  ('Amazon RDS','Database','RDS:db.r5.2xlarge Multi-AZ','Hrs',0.96,24,24,PM_DATA),
  ('Amazon RDS','Database','RDS:StorageUsage','GB-Mo',0.115,500,10000,PM_OTHER),
  # Aurora
  ('Amazon Aurora','Database','Aurora:ServerlessV2Usage','ACU-Hr',0.12,10,500,PM_DATA),
  ('Amazon Aurora','Database','Aurora:db.r5.2xlarge','Hrs',0.58,24,24,PM_DATA),
  ('Amazon Aurora','Database','Aurora:StorageUsage','GB-Mo',0.10,200,10000,PM_OTHER),
  # Redshift
  ('Amazon Redshift','Analytics','NodeUsage:dc2.8xlarge','Hrs',4.80,24,24,PM_DATA),
  ('Amazon Redshift','Analytics','NodeUsage:ra3.4xlarge','Hrs',3.26,24,48,PM_DATA),
  ('Amazon Redshift','Analytics','Serverless RPU-Hour','RPU-Hr',0.375,2,64,PM_OTHER),
  # DynamoDB
  ('Amazon DynamoDB','Database','DDB:WriteCapacityUnit-Hrs','WCU-Hr',0.00065,1000,50000,PM_OTHER),
  ('Amazon DynamoDB','Database','DDB:ReadCapacityUnit-Hrs','RCU-Hr',0.00013,5000,200000,PM_OTHER),
  ('Amazon DynamoDB','Database','TimedStorage-ByteHrs','GB-Mo',0.25,50,2000,PM_OTHER),
  # ElastiCache
  ('Amazon ElastiCache','Database','NodeUsage:cache.r6g.large','Hrs',0.166,24,48,PM_DATA),
  ('Amazon ElastiCache','Database','NodeUsage:cache.r6g.2xlarge','Hrs',0.665,24,24,PM_DATA),
  # DocumentDB
  ('Amazon DocumentDB','Database','InstanceUsage:db.r5.xlarge','Hrs',0.277,24,24,PM_DATA),
  # Neptune
  ('Amazon Neptune','Database','InstanceUsage:db.r5.xlarge','Hrs',0.295,24,24,PM_DATA),
  # Timestream
  ('Amazon Timestream','Database','TimestreamWrite-Records','1M Records',0.50,1,50,PM_OTHER),
  ('Amazon Timestream','Database','TimestreamStorage-Memory','GB-Hr',0.036,10,100,PM_OTHER),
  # EMR
  ('Amazon EMR','Analytics','EMR:m5.2xlarge','Hrs',0.096,48,192,PM_COMPUTE),
  ('Amazon EMR','Analytics','EMR:r5.4xlarge','Hrs',0.192,24,96,PM_COMPUTE),
  # Glue
  ('AWS Glue','Analytics','Glue-DPU-Hour (ETL)','DPU-Hr',0.44,2,100,PM_OTHER),
  ('AWS Glue','Analytics','Glue-DPU-Hour (Crawler)','DPU-Hr',0.44,1,20,PM_OTHER),
  # Athena
  ('Amazon Athena','Analytics','DataScannedInTB','TB',5.00,0.1,10,PM_OTHER),
  # QuickSight
  ('Amazon QuickSight','Analytics','QuickSight-User-Month','User-Mo',18.0,10,100,PM_OTHER),
  # SageMaker
  ('Amazon SageMaker','AI / ML','Training:ml.p3.2xlarge','Hrs',3.825,4,24,PM_COMPUTE),
  ('Amazon SageMaker','AI / ML','Endpoint:ml.c5.2xlarge','Hrs',0.454,24,48,PM_OTHER),
  ('Amazon SageMaker','AI / ML','Studio:ml.t3.medium','Hrs',0.046,8,24,PM_OTHER),
  # Bedrock
  ('Amazon Bedrock','AI / ML','Claude 3 Sonnet Input','1K Tokens',0.003,100,10000,PM_OTHER),
  ('Amazon Bedrock','AI / ML','Claude 3 Sonnet Output','1K Tokens',0.015,50,5000,PM_OTHER),
  ('Amazon Bedrock','AI / ML','Titan Embeddings Tokens','1K Tokens',0.0001,1000,100000,PM_OTHER),
  # Comprehend / Rekognition / Textract
  ('Amazon Comprehend','AI / ML','Comprehend-Units','Units',0.0001,1000,100000,PM_OTHER),
  ('Amazon Rekognition','AI / ML','Rekognition-Images','1K Images',1.00,1,50,PM_OTHER),
  ('Amazon Textract','AI / ML','Textract-Pages','1K Pages',1.50,0.1,10,PM_OTHER),
  ('Amazon Forecast','AI / ML','Forecast-DataPoints','1K Points',0.60,1,50,PM_OTHER),
  # Kinesis
  ('Amazon Kinesis','Integration','ShardHour','Shard-Hr',0.015,24,240,PM_OTHER),
  ('Amazon Kinesis','Integration','PUT-Payload-Unit','1M Units',0.014,100,10000,PM_OTHER),
  # MSK
  ('Amazon MSK','Integration','MSK:kafka.m5.large','Hrs',0.216,24,48,PM_DATA),
  # SNS/SQS
  ('Amazon SNS','Integration','SNS-Requests','1M Requests',0.50,0.1,10,PM_OTHER),
  ('Amazon SQS','Integration','SQS-Requests','1M Requests',0.40,1,50,PM_OTHER),
  # API Gateway
  ('Amazon API Gateway','Integration','REST API Calls','1M Calls',3.50,0.5,50,PM_OTHER),
  ('Amazon API Gateway','Integration','WebSocket Messages','1M Messages',1.00,1,100,PM_OTHER),
  # Step Functions
  ('AWS Step Functions','Integration','StateTransition','1K Transitions',0.025,10,1000,PM_OTHER),
  # Networking
  ('Amazon CloudFront','Networking','DataTransfer-Out-Bytes','GB',0.085,100,5000,PM_OTHER),
  ('Amazon CloudFront','Networking','Requests-HTTPS','10K Requests',0.010,10,1000,PM_OTHER),
  ('Amazon VPC','Networking','NatGateway-Hours','Hrs',0.045,24,24,PM_OTHER),
  ('Amazon VPC','Networking','NatGateway-Bytes','GB',0.045,10,1000,PM_OTHER),
  ('Amazon Route 53','Networking','HostedZone','Month',0.50,5,30,PM_OTHER),
  ('AWS Direct Connect','Networking','DataXfer-Out','GB',0.020,100,10000,PM_OTHER),
  ('Elastic Load Balancing','Networking','LoadBalancerUsage','Hrs',0.008,24,24,PM_OTHER),
  ('Elastic Load Balancing','Networking','LCUUsage','LCU-Hrs',0.008,10,200,PM_OTHER),
  ('AWS Data Transfer','Networking','DataTransfer-Regional-Bytes','GB',0.010,100,5000,PM_OTHER),
  # Security
  ('AWS KMS','Security','KMS-Requests','10K Requests',0.03,1,100,PM_OTHER),
  ('AWS WAF','Security','WebACL-Month','Month',5.00,1,5,PM_OTHER),
  ('Amazon Cognito','Security','MAU','MAU',0.0055,1000,50000,PM_OTHER),
  ('AWS Secrets Manager','Security','Secret-Month','Secret-Mo',0.40,10,200,PM_OTHER),
  ('AWS Shield','Security','Shield Advanced','Month',3000,1,1,PM_OTHER),
  # Management
  ('Amazon CloudWatch','Management','CW:MetricMonitorUsage','Metric-Mo',0.30,10,500,PM_OTHER),
  ('Amazon CloudWatch','Management','CW:LogsStorage','GB-Mo',0.03,50,2000,PM_OTHER),
  ('AWS CloudTrail','Management','CloudTrail-Event','100K Events',0.10,1,100,PM_OTHER),
  ('AWS Systems Manager','Management','SSM-AdvancedInstanceHour','Adv.Inst-Hr',0.00695,24,240,PM_OTHER),
  ('AWS Backup','Management','BackupStorage-AmazonS3','GB-Mo',0.05,500,10000,PM_OTHER),
  # DevOps
  ('AWS CodePipeline','DevOps','CodePipeline-Pipeline','Pipeline-Mo',1.00,5,30,PM_OTHER),
  ('AWS CodeBuild','DevOps','CodeBuild-Minute','Build-Min',0.005,100,5000,PM_OTHER),
]

# ─────────────────────────────────────────────────────────────────────────────
# GCP SKU CATALOG
# ─────────────────────────────────────────────────────────────────────────────
GCP_SKUS = [
  # Compute Engine
  ('Compute Engine','Compute','N2 Instance Core running in Americas','vCPU-Hour',0.031611,24,240,PM_COMPUTE),
  ('Compute Engine','Compute','N2 Instance Ram running in Americas','GB-Hour',0.004237,48,480,PM_COMPUTE),
  ('Compute Engine','Compute','N2D AMD Instance Core running in Americas','vCPU-Hour',0.027027,24,192,PM_COMPUTE),
  ('Compute Engine','Compute','Spot Preemptible N2 Instance Core','vCPU-Hour',0.007897,24,240,PM_COMPUTE),
  ('Compute Engine','Compute','Nvidia Tesla T4 GPU attached to Spot','GPU-Hour',0.130,4,24,PM_COMPUTE),
  ('Compute Engine','Compute','Nvidia A100 80GB GPU','GPU-Hour',2.933,2,16,PM_COMPUTE),
  ('Compute Engine','Compute','Storage PD Capacity','GB-Month',0.040,500,20000,PM_OTHER),
  # GKE
  ('Google Kubernetes Engine','Compute','Zonal Cluster Management Fee','Hour',0.10,24,24,PM_OTHER),
  ('Google Kubernetes Engine','Compute','N2 Standard 4 Node','vCPU-Hour',0.031611,96,384,PM_COMPUTE),
  # Cloud Run
  ('Cloud Run','Serverless','CPU Allocation Time','vCPU-Second',0.000024,1000,100000,PM_OTHER),
  ('Cloud Run','Serverless','Memory Allocation Time','GB-Second',0.0000025,2000,200000,PM_OTHER),
  ('Cloud Run','Serverless','Requests','1M Requests',0.40,1,100,PM_OTHER),
  # Cloud Functions
  ('Cloud Functions','Serverless','Invocations','1M Invocations',0.40,0.1,20,PM_OTHER),
  ('Cloud Functions','Serverless','Compute Time','GB-Second',0.0000025,5000,500000,PM_OTHER),
  # App Engine
  ('App Engine','Compute','B4 Instance Hours','Instance-Hour',0.05,24,96,PM_OTHER),
  ('App Engine','Compute','F4 Frontend Instance Hours','Instance-Hour',0.05,8,32,PM_OTHER),
  # Batch
  ('Batch','Compute','N2 vCPU','vCPU-Hour',0.031611,10,200,PM_COMPUTE),
  # Cloud Storage
  ('Cloud Storage','Storage','Standard Storage','GB-Month',0.020,1000,50000,PM_OTHER),
  ('Cloud Storage','Storage','Nearline Storage','GB-Month',0.010,500,20000,PM_OTHER),
  ('Cloud Storage','Storage','Class A Operations','10K Ops',0.05,10,1000,PM_OTHER),
  ('Cloud Storage','Storage','Network Egress','GB',0.12,10,1000,PM_OTHER),
  # Persistent Disk
  ('Persistent Disk','Storage','SSD backed PD Capacity','GB-Month',0.170,200,5000,PM_OTHER),
  ('Persistent Disk','Storage','Standard PD Capacity','GB-Month',0.040,500,20000,PM_OTHER),
  # Filestore
  ('Filestore','Storage','Basic HDD Capacity','GB-Month',0.200,500,5000,PM_OTHER),
  # Cloud Backup
  ('Cloud Backup','Storage','Backup Storage','GB-Month',0.023,200,10000,PM_OTHER),
  # Cloud SQL
  ('Cloud SQL','Database','DB n1-standard-4 (MySQL)','Hour',0.385,24,24,PM_DATA),
  ('Cloud SQL','Database','DB n1-highmem-8 (PostgreSQL)','Hour',0.754,24,24,PM_DATA),
  ('Cloud SQL','Database','High Availability','Hour',0.770,24,24,PM_DATA),
  ('Cloud SQL','Database','Storage Capacity SSD','GB-Month',0.170,200,5000,PM_OTHER),
  # Cloud Spanner
  ('Cloud Spanner','Database','Processing Unit','Node-Hour',0.90,24,240,PM_DATA),
  ('Cloud Spanner','Database','Storage','GB-Month',0.300,100,5000,PM_OTHER),
  # Bigtable
  ('Cloud Bigtable','Database','SSD Storage Node','Node-Hour',0.65,24,48,PM_DATA),
  ('Cloud Bigtable','Database','SSD Storage','GB-Month',0.17,100,5000,PM_OTHER),
  # Firestore
  ('Firestore','Database','Document Reads','100K Reads',0.06,10,1000,PM_OTHER),
  ('Firestore','Database','Document Writes','100K Writes',0.18,5,500,PM_OTHER),
  ('Firestore','Database','Storage','GB-Month',0.18,10,500,PM_OTHER),
  # Memorystore
  ('Memorystore','Database','Redis Basic M1','GB-Hour',0.049,24,24,PM_DATA),
  ('Memorystore','Database','Redis Standard M2','GB-Hour',0.098,24,24,PM_DATA),
  # AlloyDB
  ('AlloyDB','Database','vCPU','vCPU-Hour',0.0715,24,96,PM_DATA),
  ('AlloyDB','Database','Memory','GB-Hour',0.00875,96,384,PM_DATA),
  # BigQuery
  ('BigQuery','Analytics','Analysis','TiB',6.25,0.1,20,PM_OTHER),
  ('BigQuery','Analytics','Active Logical Storage','GB-Month',0.020,1000,100000,PM_OTHER),
  ('BigQuery','Analytics','Long Term Storage','GB-Month',0.010,5000,500000,PM_OTHER),
  ('BigQuery','Analytics','BI Engine Reservation','GB-Month',8.00,10,100,PM_OTHER),
  # Dataproc
  ('Dataproc','Analytics','Dataproc Premium n1-standard-4','vCPU-Hour',0.01,24,120,PM_COMPUTE),
  ('Dataproc','Analytics','Dataproc Preemptible n1-standard-4','vCPU-Hour',0.005,24,96,PM_COMPUTE),
  # Dataflow
  ('Dataflow','Analytics','Dataflow Shuffle','GB',0.011,100,10000,PM_OTHER),
  ('Dataflow','Analytics','vCPU Running','vCPU-Hour',0.056,10,200,PM_OTHER),
  # Looker
  ('Looker','Analytics','Developer User License','User-Month',125.0,5,20,PM_OTHER),
  ('Looker','Analytics','Standard User License','User-Month',30.0,20,100,PM_OTHER),
  # Data Fusion
  ('Cloud Data Fusion','Analytics','Basic Edition','Instance-Hour',0.35,24,24,PM_OTHER),
  # Vertex AI
  ('Vertex AI','AI / ML','Training: n1-standard-8','Node-Hour',0.380,4,48,PM_COMPUTE),
  ('Vertex AI','AI / ML','Training: a2-highgpu-1g (A100)','Node-Hour',3.673,2,16,PM_COMPUTE),
  ('Vertex AI','AI / ML','Prediction: n1-standard-4','Node-Hour',0.190,24,48,PM_OTHER),
  ('Vertex AI','AI / ML','Gemini 1.5 Pro Input Tokens','1K Tokens',0.00125,500,50000,PM_OTHER),
  ('Vertex AI','AI / ML','Gemini 1.5 Pro Output Tokens','1K Tokens',0.005,100,10000,PM_OTHER),
  ('Vertex AI','AI / ML','Model Garden Inference','1K Predictions',0.10,10,1000,PM_OTHER),
  # AI APIs
  ('Natural Language AI','AI / ML','Natural Language Units','1K Units',1.00,1,50,PM_OTHER),
  ('Vision AI','AI / ML','Label Detection Images','1K Images',1.50,0.5,20,PM_OTHER),
  ('Speech-to-Text','AI / ML','Speech Recognition','1 Minute',0.016,100,5000,PM_OTHER),
  ('Document AI','AI / ML','Document Pages','1K Pages',1.50,0.1,10,PM_OTHER),
  ('Recommendations AI','AI / ML','Prediction Requests','1K Requests',0.27,10,1000,PM_OTHER),
  # Pub/Sub
  ('Pub/Sub','Integration','Message Delivery','TiB',60.0,0.001,0.5,PM_OTHER),
  # Cloud Composer
  ('Cloud Composer','Integration','Composer 2 vCPU','vCPU-Hour',0.062,24,120,PM_OTHER),
  # Apigee
  ('Apigee API Management','Integration','API Calls','1M Calls',3.50,0.5,50,PM_OTHER),
  ('Apigee API Management','Integration','Environment Unit','Hour',0.17,24,24,PM_OTHER),
  # Eventarc / Tasks
  ('Eventarc','Integration','Events','1M Events',0.10,1,50,PM_OTHER),
  ('Cloud Tasks','Integration','Task Operations','1M Ops',0.40,0.5,20,PM_OTHER),
  # Networking
  ('Cloud Networking','Networking','Premium Tier Egress','GB',0.12,10,1000,PM_OTHER),
  ('Cloud Networking','Networking','Standard Tier Egress','GB',0.085,5,500,PM_OTHER),
  ('Cloud Load Balancing','Networking','Forwarding Rule Charge','Hour',0.025,24,24,PM_OTHER),
  ('Cloud Load Balancing','Networking','Data Processed','GB',0.008,100,5000,PM_OTHER),
  ('Cloud CDN','Networking','Cache Egress','GB',0.08,10,500,PM_OTHER),
  ('Cloud DNS','Networking','Managed Zone','Zone-Month',0.20,5,20,PM_OTHER),
  ('Cloud Interconnect','Networking','Dedicated 10Gbps Port','Month',1700.0,1,2,PM_OTHER),
  # Security
  ('Cloud Armor','Security','WAF Rule Evaluation','1M Requests',0.75,1,100,PM_OTHER),
  ('Cloud KMS','Security','Key Versions','Key-Month',0.06,20,200,PM_OTHER),
  ('Secret Manager','Security','Active Secret Versions','Month',0.06,20,200,PM_OTHER),
  ('Cloud Identity','Security','Cloud Identity Premium','User-Month',6.00,50,500,PM_OTHER),
  # Monitoring / DevOps
  ('Cloud Monitoring','Management','Monitoring Data Ingestion','MiB',0.258,100,5000,PM_OTHER),
  ('Cloud Logging','Management','Log Volume','GiB',0.50,10,500,PM_OTHER),
  ('Artifact Registry','DevOps','Storage','GB-Month',0.10,50,500,PM_OTHER),
  ('Cloud Build','DevOps','Build Minutes','Build-Min',0.003,100,5000,PM_OTHER),
  ('Firebase','Mobile/Web','Spark Plan Overages','GB',0.026,5,100,PM_OTHER),
]

# ─────────────────────────────────────────────────────────────────────────────
# Row generator
# ─────────────────────────────────────────────────────────────────────────────
NOW = datetime(2026, 4, 26)
NINETY_AGO = NOW - timedelta(days=89)
random.seed(9999)

def gen_rows(cloud, accounts, skus, start, end):
    rows = []
    cur = start
    while cur <= end:
        dow  = cur.weekday()
        wf   = 0.55 if dow >= 5 else 1.0
        mf   = 1.0 + 0.15 * (cur.day / 28)
        ds   = cur.strftime('%Y-%m-%d')
        ts   = cur.strftime('%Y-%m-%d %H:%M:%S')
        for acct_id, acct_name, region in accounts:
            rmul = REGION_MUL.get(region, 1.0)
            for svc, cat, sku_desc, unit, base_price, qmin, qmax, pm_def in skus:
                pm      = pick_pm(pm_def)
                disc    = RESERVED_DISC[pm]
                unit_p  = round(base_price * rmul * disc, 8)
                qty     = round(random.uniform(qmin, qmax) * wf * mf, 4)
                cost    = round(unit_p * qty * random.uniform(0.92, 1.08), 4)
                tags    = make_tags()
                rg      = f'rg-{acct_name.lower().replace(" ","-")}' if cloud == 'azure' else ''
                rows.append((cloud, acct_id, acct_name, ds, svc, cat,
                              sku_desc, unit, qty, unit_p, cost, 'USD',
                              pm, region, rg, tags, ts))
        cur += timedelta(days=1)
    return rows

azure_rows = gen_rows('azure', AZURE_SUBSCRIPTIONS, AZURE_SKUS, NINETY_AGO, NOW)
aws_rows   = gen_rows('aws',   AWS_ACCOUNTS,        AWS_SKUS,   NINETY_AGO, NOW)
gcp_rows   = gen_rows('gcp',   GCP_PROJECTS,        GCP_SKUS,   NINETY_AGO, NOW)
all_rows   = azure_rows + aws_rows + gcp_rows

print(f'Azure  : {len(azure_rows):>7,} rows  ({len(AZURE_SKUS)} SKUs × {len(AZURE_SUBSCRIPTIONS)} subscriptions × 90d)')
print(f'AWS    : {len(aws_rows):>7,} rows  ({len(AWS_SKUS)} SKUs × {len(AWS_ACCOUNTS)} accounts × 90d)')
print(f'GCP    : {len(gcp_rows):>7,} rows  ({len(GCP_SKUS)} SKUs × {len(GCP_PROJECTS)} projects × 90d)')
print(f'Total  : {len(all_rows):>7,} rows')
az_cost  = sum(r[10] for r in azure_rows)
aws_cost = sum(r[10] for r in aws_rows)
gcp_cost = sum(r[10] for r in gcp_rows)
print(f'\nEst. 90-day spend:')
print(f'  Azure  : ${az_cost:>12,.0f}')
print(f'  AWS    : ${aws_cost:>12,.0f}')
print(f'  GCP    : ${gcp_cost:>12,.0f}')
print(f'  Total  : ${az_cost+aws_cost+gcp_cost:>12,.0f}')


In [ ]:
TARGET = f'{CATALOG}.platform.cloud_platform_costs'

run_sql('create platform schema', f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.platform')
if RECREATE_TABLE:
    run_sql('drop cloud costs', f'DROP TABLE IF EXISTS {TARGET}')
run_sql('create cloud costs', f'''
CREATE TABLE IF NOT EXISTS {TARGET} (
    cloud_provider   STRING   NOT NULL COMMENT 'azure | aws | gcp',
    account_id       STRING   NOT NULL COMMENT 'Subscription ID / AWS Account ID / GCP Project ID',
    account_name     STRING            COMMENT 'Human-readable account name',
    usage_date       DATE     NOT NULL,
    service_name     STRING            COMMENT 'Top-level cloud service (e.g. Amazon EC2)',
    service_category STRING            COMMENT 'Compute | Storage | Database | Analytics | AI / ML | ...',
    sku_description  STRING            COMMENT 'Meter/SKU level (e.g. BoxUsage:m5.xlarge)',
    usage_unit       STRING            COMMENT 'Unit of measure (Hrs, GB-Mo, 1K Tokens, ...)',
    usage_quantity   DOUBLE            COMMENT 'Quantity consumed in this unit',
    unit_price       DOUBLE            COMMENT 'Price per unit (after reserved/spot discount)',
    cost_usd         DOUBLE            COMMENT 'usage_quantity * unit_price',
    currency         STRING,
    pricing_model    STRING            COMMENT 'ON_DEMAND | RESERVED_1Y | RESERVED_3Y | SPOT | COMMITTED_1Y',
    region           STRING,
    resource_group   STRING            COMMENT 'Azure resource group; empty for AWS/GCP',
    tags             MAP<STRING, STRING> COMMENT 'team, environment, cost_center, project',
    ingested_at      TIMESTAMP
)
USING DELTA
PARTITIONED BY (cloud_provider, usage_date)
TBLPROPERTIES (\'delta.autoOptimize.optimizeWrite\' = \'true\',
               \'delta.autoOptimize.autoCompact\'   = \'true\')
''')
print('Table ready:', TARGET)

# Build SQL value strings
def sql_esc(s): return str(s).replace("'", "''")

sql_rows = []
for r in all_rows:
    cloud, acct_id, acct_name, ds, svc, cat, sku, unit, qty, uprice, cost, curr, pm, region, rg, tags_json, ts = r
    # Parse tags JSON → SQL map literal
    import json as _j
    t = _j.loads(tags_json)
    tags_sql = "map(" + ",".join(f"'{k}','{sql_esc(v)}'" for k,v in t.items()) + ")"
    sql_rows.append(
        f"('{cloud}','{acct_id}','{sql_esc(acct_name)}','{ds}',"
        f"'{sql_esc(svc)}','{cat}',"
        f"'{sql_esc(sku)}','{unit}',{qty},{uprice},{cost},'{curr}',"
        f"'{pm}','{region}','{sql_esc(rg)}',"
        f"{tags_sql},'{ts}')"
    )

print(f'Inserting {len(sql_rows):,} rows in chunks of 300...')
for i, chunk in enumerate(chunk_list(sql_rows, 300)):
    run_sql(f'cloud cost chunk {i+1}/{(len(sql_rows)//300)+1}',
            f'INSERT INTO {TARGET} VALUES\n' + ',\n'.join(chunk))

print(f'\n✅ cloud_platform_costs loaded: {len(sql_rows):,} rows')
run_sql('verify',
    f"SELECT cloud_provider, pricing_model, COUNT(*) AS rows, "
    f"ROUND(SUM(cost_usd),0) AS total_usd "
    f"FROM {TARGET} GROUP BY 1,2 ORDER BY 1,2")
